Processa_Weekly_Infra.ipynb

Roda toda SEXTA-feira. Diferente do Processa_Daily_Infra.ipynb (1 dia),
esse agrega os documentos dos ÚLTIMOS 7 DIAS (hoje e os 6 anteriores) e
manda pro serviço externo usando o perfil semanal
(`config_infra_weekly.json`, escrito pelo gera_config.ipynb -- arquivo
PRÓPRIO, não interfere no config_N.json diário).

Esse notebook NÃO envia e-mail sozinho -- ele só gera e salva o HTML do
resumo semanal (itens + Conclusão da Semana, incluindo o ranking de
fontes). Quem monta o documento final (resumo semanal + seção de fontes
já cobertas + link pro relatório de progresso + botão de feedback) e
dispara o e-mail é o `scripts/montar_resumo_semanal_completo.py`, rodado
logo depois deste, no mesmo Job de sexta-feira.


In [0]:
import time
import json
import requests
from datetime import datetime, timedelta


In [0]:
# ---------------------------------------------------------------------------
# Config
# ---------------------------------------------------------------------------
import os

CONN_STR = os.environ.get('AZURE_STORAGE_CONNECTION_STRING')

BASE_URL = "https://ui-agents.azurewebsites.net"
UI_AGENTS_DEV_API_KEY = os.environ['UI_AGENTS_DEV_API_KEY']
USER_ID = "marcos.markevich@kinea.com.br"

DATE = datetime.today().strftime("%Y-%m-%d")   # data de referência (dia do envio, sexta)
JANELA_DIAS = 7
DATAS_DA_SEMANA = [
    (datetime.today() - timedelta(days=i)).strftime("%Y-%m-%d")
    for i in range(JANELA_DIAS)
]  # [hoje, ontem, ..., 6 dias atrás]

BASE_VOLUME_PATH = "/Volumes/desafio_kinea/research/research_volume/infraestrutura"
FILES_ROOT = os.path.join(BASE_VOLUME_PATH, "files")
OUTPUTS_ROOT = os.path.join(BASE_VOLUME_PATH, "outputs_weekly")   # pasta PRÓPRIA -- nunca colide com outputs/ do diário
PROMPTS_ROOT = os.path.join(BASE_VOLUME_PATH, "prompts")
CAMINHO_CONFIG_WEEKLY = os.path.join(PROMPTS_ROOT, "config_infra_weekly.json")

REPORT_NAME = 'Resumo Semanal - Infraestrutura'
REPORT_CODE = 'infraestrutura_desafio_weekly'
HEADERS = {
    "accept": "*/*",
    "accept-language": "pt-BR,pt;q=0.9,en-US;q=0.8,en;q=0.7",
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
        "(KHTML, like Gecko) Chrome/129.0.0.0 Safari/537.36"
    ),
}

APP_SESSION = requests.Session()


In [0]:
# ---------------------------------------------------------------------------
# Server interaction -- idêntico ao Processa_Daily_Infra.ipynb, sem mudança
# ---------------------------------------------------------------------------

def poll_until_done(agent_call_id: str, poll_every: int = 10) -> dict:
    started = time.time()
    last_progress = None
    headers = {"X-API-Key": UI_AGENTS_DEV_API_KEY}

    while True:
        response = APP_SESSION.post(
            f"{BASE_URL}/job_polling/poll_the_results",
            headers=headers,
            json={"agent_call_id": agent_call_id},
            timeout=60,
        )
        response.raise_for_status()
        data = response.json()
        status = data.get("status")
        progress = data.get("progress")
        if progress != last_progress:
            elapsed = int(time.time() - started)
            print(f"  [{elapsed}s] status={status} progress={progress}")
            last_progress = progress
        if status in ("DONE", "ERROR", "FAILED"):
            return data
        time.sleep(poll_every)


In [0]:
import re

# ---------------------------------------------------------------------------
# Volume storage helpers -- os de leitura de documento são os MESMOS do
# Processa_Daily_Infra.ipynb (mesma correção de os.walk já aplicada lá);
# só write_json_file/read_json_file/sanitize são reaproveitados sem mudança.
# ---------------------------------------------------------------------------


def write_json_file(path: str, payload: object) -> None:
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w", encoding="utf-8") as file_handle:
        json.dump(payload, file_handle, ensure_ascii=False, indent=2)


def read_json_file(path: str) -> object:
    with open(path, "r", encoding="utf-8") as file_handle:
        return json.load(file_handle)


def get_files_date_dir(date: str) -> str:
    return os.path.join(FILES_ROOT, date)


def load_documents_from_volume(date: str) -> list[dict]:
    date_dir = get_files_date_dir(date)
    if not os.path.exists(date_dir):
        print(f"  [aviso] pasta não existe (dia sem captura?): {date_dir}")
        return []

    documents: list[dict] = []
    metadata_paths = sorted(
        os.path.join(root, file_name)
        for root, _dirs, file_names in os.walk(date_dir)
        for file_name in file_names
        if file_name.endswith(".json")
    )

    for metadata_path in metadata_paths:
        file_id = os.path.basename(metadata_path)[: -len(".json")]
        text_path = metadata_path[: -len(".json")] + ".txt"
        if not os.path.exists(text_path):
            continue

        metadata = read_json_file(metadata_path)
        if not isinstance(metadata, dict):
            continue

        with open(text_path, "r", encoding="utf-8") as file_handle:
            text = file_handle.read()

        documents.append({
            "text": text,
            "source_id": metadata.get("source_id", ""),
            "title": metadata.get("title", ""),
            "description": metadata.get("description", ""),
            "url": metadata.get("url", ""),
            "date": metadata.get("date", ""),
            "published_at": metadata.get("published_at", ""),
            "file_id": f"{date}_{file_id}",  # prefixo da data -- evita colisao de file_id entre dias diferentes
        })
    return documents


def carregar_documentos_da_semana(datas: list[str]) -> list[dict]:
    todos = []
    for data in datas:
        docs_do_dia = load_documents_from_volume(data)
        print(f"  {data}: {len(docs_do_dia)} documento(s)")
        todos.extend(docs_do_dia)
    return todos


In [0]:
# ---------------------------------------------------------------------------
# Config semanal -- lê SEMPRE o arquivo de nome fixo config_infra_weekly.json
# (sem a lógica de "maior número" do diário -- não precisa, só existe um).
# ---------------------------------------------------------------------------

def render_prompt_value(value: object, today_value: str) -> object:
    if isinstance(value, str):
        return value.replace("{today}", today_value)
    return value


def build_custom_prompts_payload_weekly() -> dict:
    today_value = datetime.strptime(DATE, "%Y-%m-%d").strftime("%A, %b %d, %Y")
    config_payload = read_json_file(CAMINHO_CONFIG_WEEKLY)
    return {
        key: render_prompt_value(value, today_value)
        for key, value in config_payload.items()
    }


def submit_report_weekly(documents: list[dict]) -> str:
    if not UI_AGENTS_DEV_API_KEY:
        raise ValueError("Fill UI_AGENTS_DEV_API_KEY before running this notebook.")

    headers = {"X-API-Key": UI_AGENTS_DEV_API_KEY}
    custom_prompts = build_custom_prompts_payload_weekly()
    custom_prompts["country_code"] = REPORT_CODE
    payload = {
        "documents": documents,
        "profile": "custom",
        "country_code": REPORT_CODE,
        "name": REPORT_NAME,
        "language": "pt",
        "custom_prompts": custom_prompts,
        "day_label": f"{REPORT_NAME} {DATE}",
        "user_id": USER_ID,
    }
    response = APP_SESSION.post(
        f"{BASE_URL}/report_agent/start_report",
        headers=headers,
        json=payload,
        timeout=60,
    )
    response.raise_for_status()
    return response.json()["agent_call_id"]


In [0]:
# ---------------------------------------------------------------------------
# Execução
# ---------------------------------------------------------------------------

os.makedirs(OUTPUTS_ROOT, exist_ok=True)

print(f"Agregando documentos dos últimos {JANELA_DIAS} dias: {DATAS_DA_SEMANA[-1]} a {DATAS_DA_SEMANA[0]}\n")
documents = carregar_documentos_da_semana(DATAS_DA_SEMANA)
print(f"\nTotal de documentos da semana: {len(documents)}")

if not documents:
    raise ValueError(f"Nenhum documento encontrado nos últimos {JANELA_DIAS} dias. Confere se os dispatchers rodaram.")

try:
    agent_call_id = submit_report_weekly(documents)
except requests.exceptions.HTTPError as e:
    print("=== CORPO DA RESPOSTA DE ERRO ===")
    print(e.response.text)
    raise

print(f"agent_call_id = {agent_call_id}")
print("[1/2] Submitting to /report_agent/start_report ... OK")
print("[2/2] Polling /job_polling/poll_the_results ...")
resultado = poll_until_done(agent_call_id)

status = resultado.get("status")
print(f"Status: {status}")

if status != "DONE":
    raise RuntimeError(f"Resumo semanal não concluído: {resultado}")

html = resultado.get("report") or resultado.get("text") or ""
print(f"HTML length: {len(html)}")


In [0]:
# ---------------------------------------------------------------------------
# Salva o HTML bruto do resumo semanal -- num caminho de nome FIXO (não
# datado), para que montar_resumo_semanal_completo.py sempre encontre a
# versão mais recente sem precisar adivinhar a data.
# ---------------------------------------------------------------------------

CAMINHO_RESUMO_SEMANAL_BRUTO = os.path.join(OUTPUTS_ROOT, "resumo_semanal_bruto.html")

with open(CAMINHO_RESUMO_SEMANAL_BRUTO, "w", encoding="utf-8") as f:
    f.write(html)

# Cópia com data, para histórico (não é lida por nenhum outro script)
caminho_historico = os.path.join(OUTPUTS_ROOT, f"resumo_semanal_{DATE}.html")
with open(caminho_historico, "w", encoding="utf-8") as f:
    f.write(html)

print(f"[ok] Resumo semanal salvo em: {CAMINHO_RESUMO_SEMANAL_BRUTO}")
print(f"[ok] Cópia histórica: {caminho_historico}")
print("Próximo passo: rodar scripts/montar_resumo_semanal_completo.py")
